# TB Portals - DA-MoE **mode = `a3`**

Mixture of direct-Timika experts. The MoE upgrade of Kantipudi A3 (no cavity agent).

**Anchor vs the locked baselines** (`baseline_runs/BASELINE_COMPARISON.md`) — Timika MAE per country:
- A2: Romania 20.11 / **Moldova 30.68** / Kazakhstan 21.35
- A3: Romania 20.26 / **Moldova 26.16** / Kazakhstan 21.90
- A1: Romania 26.84 / **Moldova 32.76** / Kazakhstan 21.87

Moldova is the target.

**Attach datasets:** `tb-portals-cxr-pngs`, `medsam-vit-b`.

## 0 - Clone the codebase  *(restart kernel after any pull that changed .py)*

In [1]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + "/scripts"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)
# After a git pull that changed .py modules, RESTART the kernel so Python reloads them.

Cloning into '/kaggle/working/dl-project-codebase'...


repo ready at /kaggle/working/dl-project-codebase


Updating files: 100% (446/446), done.


## Install deps

In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.4 MB/s eta 0:00:00


deps installed


## Paths - edit dataset slugs if yours differ

In [3]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))

KAGGLE_EXPORT: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export -> True
MEDSAM_CKPT:   /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth -> True
LUNG_DECODER:  /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt -> True


## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [4]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
Paper manifest: 5010 images (target 5010) -> /kaggle/working/tbportals_manifest_paper.csv


## 2 - MedSAM lung crops (~25 min first time; idempotent)

In [5]:
import os, sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from cache_lung_crops import main as crops_main
argv = ["--manifest", PAPER_MANIFEST, "--out-dir", CROPS_DIR,
        "--medsam-ckpt", MEDSAM_CKPT, "--size", "224", "--pad", "32"]
if os.path.isfile(LUNG_DECODER): argv += ["--lung-decoder-ckpt", LUNG_DECODER]
crops_main(argv)
print("crops ->", CROPS_DIR, "| count:", len(os.listdir(CROPS_DIR)))

[crops] device=cuda:Tesla T4


[crops] 0/5010 cached; generating the remaining 5010.


[crops] loaded fine-tuned lung decoder: /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt


[crops] 200/5010 (lung=200, fallback=0)


[crops] 400/5010 (lung=400, fallback=0)


...

[crops] 4400/5010 (lung=4400, fallback=0)


[crops] 4600/5010 (lung=4600, fallback=0)


[crops] 4800/5010 (lung=4800, fallback=0)


[crops] 5000/5010 (lung=5000, fallback=0)


[crops] DONE -> /kaggle/working/crops (lung-cropped=5010, whole-image fallback=0)
crops -> /kaggle/working/crops | count: 5010


## 3 - Configure

In [6]:
# ---- this notebook is dedicated to MODE = "a3" --------------------------
MODE     = "a3"
SEEDS    = ["0", "1", "2"]   # full run; use ["0"] first if you want a fast sanity check
EPOCHS   = "30"
PRETRAIN = "10"              # phase-1 expert-pretraining epochs (rest = gate+critic+DANN)
OUT_DIR  = f"/kaggle/working/checkpoints/moe_{MODE}"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("will run MoE mode =", MODE, "seeds =", SEEDS, "->", OUT_DIR)

will run MoE mode = a3 seeds = ['0', '1', '2'] -> /kaggle/working/checkpoints/moe_a3


## 4 - Train + evaluate

Per (country, seed): frozen cavity agent (if used) -> MoE (phase 1 experts -> phase 2 gate+critic+DANN). ~2-3 h for 3 seeds.

In [7]:
from src.training.train_da_moe import main as moe_main
argv = ["--mode", MODE, "--manifest", PAPER_MANIFEST, "--crops-dir", CROPS_DIR,
        "--out-dir", OUT_DIR, "--held-outs", "Romania", "Moldova", "Kazakhstan",
        "--seeds", *SEEDS, "--epochs", EPOCHS, "--pretrain-epochs", PRETRAIN,
        "--batch-size", "60", "--accum-steps", "5", "--num-workers", "2"]
# a1/a2/fusion use the cavity agent on whole images (matches the locked A2 config):
if MODE in ("a1", "a2", "fusion"):
    argv += ["--cavity-no-lung-crop"]
moe_main(argv)

[da-moe] device=cuda mode=a3 dann=True critic=True

===== DA-MoE[a3]  Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[MoE] reg_train=3832 val=958 test=220 | 7 train-countries for DANN
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


  0%|          | 0.00/30.8M [00:00<?, ?B/s]

 25%|██▍       | 7.62M/30.8M [00:00<00:00, 79.6MB/s]

 83%|████████▎ | 25.5M/30.8M [00:00<00:00, 143MB/s] 

100%|██████████| 30.8M/30.8M [00:00<00:00, 142MB/s]

...

  [P2] epoch 27 train_loss=1.64811 val_timika_mse=0.05809 grl_lambda=1.000


  [P2] epoch 28 train_loss=1.63086 val_timika_mse=0.05977 grl_lambda=1.000


  [P2] epoch 29 train_loss=1.63882 val_timika_mse=0.05753 grl_lambda=1.000


[RESULT] MoE-a3 Kazakhstan seed=2  Timika_MAE=32.35 (23.11%) | paper 22.12   Pearson=0.22 | paper 0.74



[da-moe] mean +/- std across seeds (vs Kantipudi A3):
  Romania      Timika_MAE=29.58+/-0.69 (paper 19.67)  Pearson=0.17 (paper 0.70)
  Moldova      Timika_MAE=36.46+/-0.48 (paper 18.98)  Pearson=0.02 (paper 0.85)
  Kazakhstan   Timika_MAE=32.62+/-0.23 (paper 22.12)  Pearson=0.18 (paper 0.74)

[da-moe] results -> /kaggle/working/checkpoints/moe_a3/results_moe.csv


## 5 - Save results + trained models (download both)

In [8]:
import os, shutil
# results CSV alone (small — quick to grab)
csv_dst = f"/kaggle/working/results_moe_{MODE}.csv"
shutil.copy(f"{OUT_DIR}/results_moe.csv", csv_dst)
# trained models + results -> one zip (the moe_*.pt checkpoints live in OUT_DIR)
zip_path = shutil.make_archive(f"/kaggle/working/checkpoints_moe_{MODE}", "zip", OUT_DIR)
n_ckpt = len([f for f in os.listdir(OUT_DIR) if f.endswith(".pt")])
print("Saved:")
print("  ", csv_dst, " (results only)")
print("  ", zip_path, f" ({n_ckpt} trained .pt models + results_moe.csv)")
print("Download BOTH from the Output panel. Drop the CSV into baseline_runs/MoE/.")

Saved:
   /kaggle/working/results_moe_a3.csv  (results only)
   /kaggle/working/checkpoints_moe_a3.zip  (9 trained .pt models + results_moe.csv)
Download BOTH from the Output panel. Drop the CSV into baseline_runs/MoE/.


## 6 - Ablations (optional, run last)

In [9]:
# ---- ablations (run last, after the full model beats the baseline) ----------
from src.training.train_da_moe import main as moe_main
def run(tag, extra):
    import os
    out = f"/kaggle/working/checkpoints/moe_{MODE}_abl_{tag}"
    os.makedirs(out, exist_ok=True)
    base = ["--mode", MODE, "--manifest", PAPER_MANIFEST, "--crops-dir", CROPS_DIR,
            "--out-dir", out, "--held-outs", "Romania", "Moldova", "Kazakhstan",
            "--seeds", "0", "--epochs", EPOCHS, "--pretrain-epochs", PRETRAIN,
            "--batch-size", "60", "--accum-steps", "5", "--num-workers", "2"]
    if MODE in ("a1", "a2", "fusion"): base += ["--cavity-no-lung-crop"]
    if "DET_ALP_CSV" in globals(): base += ["--det-alp-csv", DET_ALP_CSV]
    print("\n==== ABLATION", MODE, tag, extra, "===="); moe_main(base + extra)

run("no_dann",   ["--no-dann"])
run("no_critic", ["--no-critic"])
run("no_both",   ["--no-dann", "--no-critic"])


==== ABLATION a3 no_dann ['--no-dann'] ====


[da-moe] device=cuda mode=a3 dann=False critic=True



===== DA-MoE[a3]  Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[MoE] reg_train=3832 val=958 test=220 | 7 train-countries for DANN


  [P1] epoch 00 train_loss=0.05686 val_timika_mse=0.04578 grl_lambda=0.000


  [P1] epoch 01 train_loss=0.03686 val_timika_mse=0.08002 grl_lambda=0.000


...

  [P2] epoch 21 train_loss=0.01364 val_timika_mse=0.04485 grl_lambda=0.000


  [P2] epoch 22 train_loss=0.01303 val_timika_mse=0.03286 grl_lambda=0.000


  [P2] epoch 23 train_loss=0.01168 val_timika_mse=0.03286 grl_lambda=0.000


  [P2] epoch 24 train_loss=0.01061 val_timika_mse=0.03415 grl_lambda=0.000


  [P2] epoch 25 train_loss=0.00960 val_timika_mse=0.03360 grl_lambda=0.000
